In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import operator
import os

In [12]:
load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [20]:
class EvaluateSchema(BaseModel):

    feedback: str = Field(description='detailed feedback of essay')
    score: int = Field(description="give the STRICT integer score for essay out of 10", ge=0, le=10)

In [21]:
structured_model = model.with_structured_output(EvaluateSchema)

In [27]:
essay = """India and AI Time

Now world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.

India have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.

In farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.

But problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.

One more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.

India must all people together – govern, school, company and normal people. We teach AI and make sure AI not bad. Also talk to other country and learn from them.

If India use AI good way, we become strong, help poor and make better life. But if only rich use AI, and poor no get, then big bad thing happen.

So, in short, AI time in India have many hope and many danger. We must go right road. AI must help all people, not only some. Then India grow big and world say "good job India"."""

In [23]:
class UPSCstate(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [24]:
def language_feedback(state: UPSCstate):
    prompt = f'evaluate the below essay strictly on basis of language and grammer only \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'language_feedback' : output.feedback, 'individual_scores' : [output.score]}

def analysis_feedback(state: UPSCstate):
    prompt = f'evaluate the below essay strictly on basis of analysis only \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback' : output.feedback, 'individual_scores' : [output.score]}

def thought_feedback(state: UPSCstate):
    prompt = f'evaluate the below essay strictly on basis of depth of thought only \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'clarity_feedback' : output.feedback, 'individual_scores' : [output.score]}

def overall_feedback(state: UPSCstate):
    prompt = f'give a summary feedback based on following feedbacks \n 1. language feedback - {state["language_feedback"]} \n 2. analysis_feedback - {state["analysis_feedback"]} 3. thought_feedback - {state["clarity_feedback"]}'
    output = model.invoke(prompt)

    avg_score = sum(state['individual_scores'])/3

    return {'overall_feedback' : output, 'avg_score' : avg_score}

In [25]:
graph = StateGraph(UPSCstate)

graph.add_node('language_feedback', language_feedback)
graph.add_node('analysis_feedback', analysis_feedback)
graph.add_node('thought_feedback', thought_feedback)
graph.add_node('overall_feedback', overall_feedback)

graph.add_edge(START, 'language_feedback')
graph.add_edge(START, 'analysis_feedback')
graph.add_edge(START, 'thought_feedback')

graph.add_edge('language_feedback', 'overall_feedback')
graph.add_edge('analysis_feedback', 'overall_feedback')
graph.add_edge('thought_feedback', 'overall_feedback')

graph.add_edge('overall_feedback', END)

workflow = graph.compile()

In [28]:
workflow.invoke({'essay' : essay})

{'essay': 'India and AI Time\n\nNow world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.\n\nIndia have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.\n\nIn farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.\n\nBut problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.\n\nOne more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.\n\n